## Initialization

### Imports/Constants

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
import pandas as pd
import numpy as np
from pint import Quantity
import bottleneck

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn,
    EnergyColumn,
    get_df_col
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
# from data_processing.processing.slice_fitting import (
#     get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
# from data_processing.processing.calibration import Detector, recalibrate
# from data_processing.processing.neutron_classification import classify
from data_processing import processing as proc
from data_processing import types as proc_types
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams,
    NumberedSlice,
    SliceFitResult,
    FitResult,
    FitErrorResult
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.processing.slice_fitting.slice_fitters import SliceFitter
from data_processing.processing.figure_of_merit import FOM
from data_processing.processing.slice_fitting.helpers import split_params, unpack_slice_fit_pool_results
from data_processing.processing.slice_fitting.bimodal_fitting import get_bimodal_fit, get_bimodal_fit_guess
from data_processing.dataframe_validation import FIT_COLUMN_NAMES, FIT_ERROR_COLUMN_NAMES
from data_processing.helpers import get_midpoints_from_bins
# from multiprocessing.pool import Pool
from joblib import Parallel, delayed

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

### Functions

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = get_df_col(df, DetectorDataframeColumn.ENERGY)
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

In [ ]:
class PeakFinderSliceFitter(SliceFitter):
    def __call__(self, numbered_slice: NumberedSlice) -> SliceFitResult:
        i, slice = numbered_slice
        slice_left_edge = self.energy_bin_edges[i]
        slice_right_edge = self.energy_bin_edges[i + 1]

        guess = get_bimodal_fit_guess(self.psd_bin_midpoints, slice)
        try:
            gamma_params, neutron_params, cov = get_bimodal_fit(
                self.psd_bin_midpoints, slice, guess=guess
            )
        except RuntimeError:
            fit_result = FitResult(
                i, None, None, slice_left_edge, slice_right_edge, None
            )
            fit_error_result = FitErrorResult(
                i, None, None, slice_left_edge, slice_right_edge
            )
            return fit_result, fit_error_result

        fom = FOM(*gamma_params[:-1], *neutron_params[:-1])
        perr = BimodalParams(*np.sqrt(np.diag(cov)))
        fit_result = FitResult(
            i, gamma_params, neutron_params, slice_left_edge, slice_right_edge, fom
        )
        fit_error_result = FitErrorResult(
            i, *split_params(perr), slice_left_edge, slice_right_edge
        )

        return fit_result, fit_error_result


class SliceFitterFactory:
    def make_slice_fitter(
        self,
        psd_bin_midpoints: np.ndarray,
        energy_bin_edges: np.ndarray,
    ) -> SliceFitter:
        return PeakFinderSliceFitter(
            psd_bin_midpoints, energy_bin_edges, None, None
        )


def scan_histogram_slices(
    histogram: np.ndarray,
    energy_bin_edges: np.ndarray,
    psd_bin_edges: np.ndarray,
    start_idx: int = 0,
    end_idx: int | None = None,
    cores: int = 4,
    use_chunks: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Determines bimodal fit and FOM for every energy slice
    in a 2D PSD/Energy histogram

    Parameters
    ----------
    histogram: ndarray
        2D PSD/Energy histogram.
        The histogram shape should be [N, M], where N is the number of energy bins, and M is the number of PSD bins.
        This matches the output of Numpy's histogram2d function.
    energy_bin_edges: ndarray
        Edge values for each energy bin in the histogram.
        For N energy bins, there must be N+1 edges.
    psd_bin_edges: ndarray
        Edge values for each PSD bin in the histogram.
        For M PSD bins, there must be M+1 edges.
    fit_style: SliceFitStyle
        The style of best fit to use
    default_bounds: BimodalBounds
        Default lower and upper bounds of fit parameters
    bounds: list[tuple[tuple[int, int], BimodalBounds]] | None, default None
        Allows custom bounds for slice ranges.
        Each list entry must have a tuple of start and stop indexes, and corresponding fit bounds.
        Bounds are used when the slice index falls within the start/stop range (start inclusive, stop exclusive).
        If index ranges overlap, the last matching range is used.
        If bounds is None, only default_bounds are used.
    start_idx: int, default 0
        Starting index (inclusive) of slice range to fit to bimodal
    end_idx: int | None, default None
        Ending index (exclusive) of slice range to fit to bimodal
    cores: int, default 4
        Number of logical cores present on this computer.
        Used to control parallelization of the scan.
    use_chunks: bool, default False
        Whether to split slices into larger chunks during parallelization.
        This can help speed up the scan on larger histograms.

    Returns
    -------
    fit_dataframe: DataFrame
        DataFrame of fit parameters including FOM (as columns) for each slice (as rows)
    error_dataframe: DataFrame
        DataFrame of (1 standard deviation) errors in fit parameters (as columns) for each slice (as rows)
    """
    end_idx = len(histogram) if end_idx is None else min(len(histogram), end_idx)
    pool_size = max(
        2 * cores, 4
    )  # based on https://jupyter-tutorial.readthedocs.io/en/stable/performance/multiprocessing.html

    # psd_bin_left_edges = psd_bin_edges[:-1]
    # psd_bin_right_edges = psd_bin_edges[1:]
    # psd_bin_centers = (psd_bin_right_edges + psd_bin_left_edges) / 2
    psd_bin_centers = get_midpoints_from_bins(psd_bin_edges)
    energy_bin_edges_limited = energy_bin_edges[start_idx : end_idx + 1]

    energy_slices = list(histogram[start_idx:end_idx, :])

    if use_chunks:
        chunksize, extra = divmod(len(energy_slices), pool_size * 4)
        if extra > 0:
            chunksize += 1
    else:
        chunksize = None

    # pool = Pool(pool_size)
    slice_fitter_factory = SliceFitterFactory()
    slice_fitter = slice_fitter_factory.make_slice_fitter(
        psd_bin_centers, energy_bin_edges_limited
    )
    # slice_fitter_factory = TestSliceFitterFactory()
    # slice_fitter = slice_fitter_factory.make_slice_fitter(0)
    # results = pool.imap_unordered(
    #     slice_fitter,
    #     enumerate(energy_slices),
    #     chunksize=chunksize,
    # )
    # results = [slice_fitter(num_slice) for num_slice in enumerate(energy_slices)]
    chunksize = chunksize if chunksize is not None else "auto"
    results = Parallel(n_jobs=pool_size, batch_size=chunksize)(delayed(slice_fitter)(x) for x in enumerate(energy_slices))

    slice_params, slice_err = unpack_slice_fit_pool_results(results)

    df = pd.DataFrame(slice_params, columns=FIT_COLUMN_NAMES)
    err_df = pd.DataFrame(slice_err, columns=FIT_ERROR_COLUMN_NAMES)

    return df, err_df

In [ ]:
def two_point_inv_lerp(y: float, p1: tuple[float, float], p2: tuple[float, float]) -> float:
    deltas = tuple([n2 - n1 for n1, n2 in zip(p1, p2)])
    m = deltas[1] / deltas[0]
    x1, y1 = p1
    if m == 0:
        return x1
    x = (y - y1) / m + x1
    return x


def calculate_q_fom_critical(
    fit_df: pd.DataFrame,
    # q_limits: tuple[float, float] | None = None
    # q_limit_hi: float | None = None
) -> tuple[float | None, float | None]:
    fom_crit = 1.27
    # if q_limits is None:
    #     q_limits = (2500, 60000)  # x axis area with clean FOM curve
    # if q_limit_hi is None:
    #     q_limit_hi = 60000
    fom_data = fit_df["fom"]
    slice_energy_min = fit_df["slice_energy_min"]
    slice_energy_max = fit_df["slice_energy_max"]
    slice_energy_mid = (slice_energy_min + slice_energy_max) / 2
    fom_x = slice_energy_mid.values
    fom_y = fom_data.values
    delta_y = fom_y[2:] - fom_y[:-2]
    stable_end_idx = np.argmax(fom_x >= 1000)

    # use moving average of delta_y to find stable region (delta_y <= threshold)
    # find first cross in stable region
    window = 5
    # bottleneck window functions use look-behind windows and fill missing with nan
    # so first window-1 values are always nan; we need to convert to look-ahead
    # 
    delta_y_mov_max = bottleneck.move_max(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    delta_y_mov_min = bottleneck.move_min(
        delta_y[:stable_end_idx+window-1], window
    )[window-1:]
    stable_max_delta = delta_y_mov_max < 0.25
    stable_min_delta = delta_y_mov_min > -0.25
    stable_delta = (stable_max_delta & stable_min_delta)
    if not stable_delta.any():
        return None, None
    stable_start_idx = np.argmax(stable_delta)
    stable_start_x = fom_x[stable_start_idx]
    cross_search_slice = fom_y[stable_start_idx:stable_end_idx]
    if not (cross_search_slice >= fom_crit).any():
        return None, stable_start_x
    # argmax gets i in slice, need to add slice start to get i in original list
    fom_critical_idx = np.argmax(cross_search_slice >= fom_crit) + stable_start_idx
    
    # possible_crosses = np.where((fom_y[1:] >= 1.27) & (fom_y[:-1] <= 1.27))[0]
    # # dd_sums = []
    # fom_critical_idx = None
    # for possible_cross in possible_crosses:
    #     pass  # STUB
        # slice_lo = possible_cross - 3 if possible_cross - 3 >= 0 else 0
        # slice_hi = possible_cross + 2
        # deltas = delta_y[slice_lo:slice_hi]
        # delta_deltas = deltas[1:] - deltas[:-1]
        # dd_sum = abs(delta_deltas).sum()
        # dd_sums.append(dd_sum)
    
    # idx_best_cross = np.argmin(np.nan_to_num(dd_sums, nan=np.inf))
    # fom_critical_idx = possible_crosses[idx_best_cross] + 1
    if fom_critical_idx is None:
        q_fom_critical = None
    elif fom_critical_idx > 0:
        # x_crit_bounds = tuple([fom_x[fom_critical_idx+x] for x in [-1, 0]])
        # y_crit_bounds = tuple([fom_y[fom_critical_idx+x] for x in [-1, 0]])
        p1 = fom_x[fom_critical_idx-1], fom_y[fom_critical_idx-1]
        p2 = fom_x[fom_critical_idx], fom_y[fom_critical_idx]
        if p1[1] > fom_crit:  # we can't make lerp extrapolate!
            q_fom_critical = None
        else:
            q_fom_critical = two_point_inv_lerp(fom_crit, p1, p2)
    else:
        q_fom_critical = fom_x[fom_critical_idx]
    return q_fom_critical, stable_start_x

## Experiment ID Input

In [ ]:
experiment_ids = ["TB-26"]

In [ ]:
# more here?
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
# calib_input = get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
# settings = get_nasa_generation_settings(calib_key)
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

## Data Loading and Initial Processing

### Data Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    psd_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    print(psd_df["ENERGY"].max())

In [ ]:
# Generate histogram

adc_width = 10
# overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
# overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_adc_histogram(
        psd_report,
        adc_width=adc_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    # exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fit_df, fit_error_df = scan_histogram_slices(
        exp_data[ExperimentDataKey.PSD_HISTOGRAM],
        exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES],
        exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    )
    # print(fit_df.head())
    # print(fit_error_df.head())
    exp_data["fit_df"] = fit_df
    exp_data["fit_error_df"] = fit_error_df

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fit_df = exp_data["fit_df"]
    print(fit_df)

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fit_df = exp_data["fit_df"]
    q_fom_critical = calculate_q_fom_critical(fit_df)
    print(q_fom_critical)
    exp_data["q_fom_critical"] = q_fom_critical

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fit_df = exp_data["fit_df"]
    q_fom_critical, _ = exp_data["q_fom_critical"]
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    
    e_min_below_critical = fit_df["slice_energy_min"] <= q_fom_critical
    e_max_above_critical = fit_df["slice_energy_max"] > q_fom_critical
    q_row_idx = fit_df[e_min_below_critical & e_max_above_critical].iloc[-1].name

    stride = 5
    count = 2
    subset_start = q_row_idx - (stride * count)
    subset_end = q_row_idx + (stride * (count + 1))
    
    slices = fit_df.iloc[subset_start:subset_end:stride]
    subset_Z = Z[subset_start:subset_end:stride, :]
    x_midpoints = (xe[1:] + xe[:-1]) / 2
    subset_x_mids = x_midpoints[subset_start:subset_end:stride]

    exp_data["crit_slice_fits"] = slices
    exp_data["subset_Z"] = subset_Z
    exp_data["subset_x_mids"] = subset_x_mids

### Plotting

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    fit_df = exp_data["fit_df"]
    q_fom_critical, search_start_x = exp_data["q_fom_critical"]
    fom_data = fit_df["fom"]
    slice_energy_min = fit_df["slice_energy_min"]
    slice_energy_max = fit_df["slice_energy_max"]
    slice_energy_mid = (slice_energy_min + slice_energy_max) / 2
    fom_x = slice_energy_mid.values
    fom_y = fom_data.values
    delta_y = fom_y[2:] - fom_y[:-2]
    delta_x = fom_x[1:-1]
    
    fig, ax = plt.subplots(figsize=(14,12))
    ax_right = ax.twinx()
    ax.plot(fom_x, fom_y, ".-")
    ax_right.plot(delta_x, delta_y, ".:")
    ax.hlines([1.27], 0, 140000, linestyles="dashed")
    ax.vlines([q_fom_critical], 0, 5, linestyles="dashed", alpha=0.5)
    ax.vlines([search_start_x], 0, 5, linestyles="dashdot", alpha=0.5)
    ax.set_xlabel("Slice $Q_{long}$ (ADC channels)", fontsize=fontsize)
    ax.set_ylabel("FOM", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    ax_right.set_ylabel("Delta FOM", fontsize=fontsize)
    ax_right.tick_params(labelsize=fontsize)
    ax.set_ylim(0, 5)
    # ax.set_xlim(2000, 60000)
    ax.set_xlim(0, 300)
    ax_right.set_ylim(-5, 5)

In [ ]:
# bg_dict = experiment_neutron_data["background"]
# edges = bg_dict["edges"]
# midpoints = (edges[1:] + edges[:-1]) / 2
# x = np.linspace(0., 4000., len(midpoints))
figsize = (15, 12)
angle_elev = 30
angle_rot = 60

def polygon_under_graph(x, y):
    """
    Construct the vertex list which defines the polygon filling the space under
    the (x, y) line graph. This assumes x is in ascending order.
    """
    return [(x[0], 0.), *zip(x, y), (x[-1], 0.)]

for exp_id, exp_data in experiment_neutron_data.items():
    slices = exp_data["crit_slice_fits"]
    subset_Z = exp_data["subset_Z"]
    e_midpoints = exp_data["subset_x_mids"]
    subset_psd_edges = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]

    psd_midpoints = (subset_psd_edges[1:] + subset_psd_edges[:-1]) / 2
    slice_fits = list(slices.iterrows())

    all_verts = []
    all_colors = []

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "3d"})
    ax.view_init(angle_elev, angle_rot)
    
    for row, slice_fit, e_val in zip(subset_Z, slice_fits, e_midpoints):
        _, slice_fit = slice_fit
        slice_fom = slice_fit["fom"]

        good_fom = slice_fom > 1.27
        plot_es = np.full(shape=psd_midpoints.shape, fill_value=e_val, dtype=float)
        exp_verts = polygon_under_graph(psd_midpoints, row)

        all_verts.append(exp_verts)

        lines = ax.plot(psd_midpoints, plot_es, row, color="green" if good_fom else "#cb3333")
        colors = [line.get_color() for line in lines]
        all_colors.extend(colors)

        annot_psd = psd_midpoints[25]
        annot_z = row[25]
        annot_text = f"{slice_fom:.3f}"
        ax.text(annot_psd - 0.01, e_val, annot_z, annot_text, fontsize=fontsize-4)

    # ax.text(0, 200, 0, "test", fontsize=fontsize)
    # ax.plot(0, 200, 0, "o", markersize=5)

    poly = mpl.collections.PolyCollection(all_verts, facecolors=all_colors, alpha=0.7)
    ax.add_collection3d(poly, zs=e_midpoints, zdir="y")

    # defaults
    ax.tick_params(labelsize=fontsize, pad=0)
    for axis3d in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis3d.labelpad = 16
        axis3d.set_rotate_label(True)
        
    # ax.xaxis.set_rotate_label(True)
    # ax.yaxis.set_rotate_label(True)
    ax.set_xlabel("PSD", fontsize=fontsize)
    ax.set_ylabel("Energy (ADC channel)", fontsize=fontsize)
    ax.set_zlabel("Counts (x1000)", fontsize=fontsize)
    
    ax.yaxis.set_major_locator(plt.MaxNLocator(5))
    ax.zaxis.set_major_locator(plt.MaxNLocator(5))
    ax.zaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x / 1000:.2f}"))

    ax.tick_params(axis="y", pad=2)
    ax.yaxis.labelpad = 24
    ax.zaxis.labelpad = 26
    
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("top")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("center")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("top")

    ax.set_box_aspect(None, zoom=0.85)